In [1]:
import pandas as pd
import os
import re
import math
import multiprocessing
cores = multiprocessing.cpu_count()
from IPython.display import clear_output

# Stopwords & punctuation
import string
from nltk.corpus import stopwords

from nltk.tokenize import word_tokenize

# Vectorization
from gensim.models.doc2vec import TaggedDocument, Doc2Vec
from gensim.test.test_doc2vec import ConcatenatedDoc2Vec

# Classification
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Naive Bayes (GaussianNB)
from sklearn.naive_bayes import GaussianNB

# K plus proches voisins (kNN)
from sklearn.neighbors import KNeighborsClassifier

# Machines à vecteurs de support (SVM)
from sklearn.svm import LinearSVC

# Régression Logistique
from sklearn.linear_model import LogisticRegression

# Forêt aléatoire 
from sklearn.ensemble import RandomForestClassifier

# Perceptron multicouches
from sklearn.neural_network import MLPClassifier

# Validation croisée - 5-fold cross-validation
stratified_kfold = StratifiedKFold(shuffle=True, random_state=42)

In [2]:
punct = string.punctuation
punct += '’'

stopw = stopwords.words('english')
for w in stopw:
    w_nopunct = re.sub(f'[{punct}]', '', w)
    if w_nopunct not in stopw:
        stopw.append(w_nopunct)

stopw += ["'d", "'ll", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would']

def tokenize_remove_stopwords(text):
 for token in word_tokenize(text):
    if token in stopw: continue
    yield (token)

**Charger les données nettoyées**

In [3]:
folder = '../2-scripts/1-preprocessing'
datasets = os.listdir(folder)
datasets

['cleaned_train_dataset_10pc.xlsx',
 'cleaned_train_dataset_20pc.xlsx',
 'cleaned_train_dataset_30pc.xlsx',
 'cleaned_train_dataset_40pc.xlsx',
 'cleaned_train_dataset_50pc.xlsx',
 'cleaned_train_dataset_60pc.xlsx',
 'cleaned_train_dataset_70pc.xlsx',
 'cleaned_train_dataset_80pc.xlsx',
 'cleaned_train_dataset_90pc.xlsx']

**Entraîner les modèles et sortir les résultats**

In [4]:
for dataset in datasets:
    report = []
    ratio = int(dataset[22:-7])
    df = pd.read_excel(os.path.join(folder, dataset))

    corpus = df['cleaned_text_post'].astype('str')
    corpus = [list(tokenize_remove_stopwords(doc)) for doc in corpus]
    corpus = [
        TaggedDocument(words, ['{}'.format(idx)])
        for idx, words in enumerate(corpus)
    ]

    # Valeurs à tester pour le nombre de traits discriminants
    n_features_values = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000] # À discuter avec Dominic , 2500, 5000, 10000, 15000]

    # Algorithmes à tester
    algorithms = {
        'Naive Bayes' : GaussianNB(), 
        'kNN (k=1)' : KNeighborsClassifier(n_neighbors=1),
        'kNN (k=2)' : KNeighborsClassifier(n_neighbors=2),
        'kNN (k=3)' : KNeighborsClassifier(n_neighbors=3),
        'Support Vector Machines (SVM)' : LinearSVC(dual='auto'),
        'Logistic Regression': LogisticRegression(n_jobs = cores),
        # 'Random Forest': RandomForestClassifier(n_jobs = cores),
        # 'Multilayered Perceptron': MLPClassifier(max_iter = 500)
    }

    for n_features in n_features_values:
        print(str(int(n_features/2)))
        model_dbow = Doc2Vec(
            corpus,
            dm = 0, # DBOW Algorithm
            vector_size= int(n_features/2),
            dm_concat = 0,
            dm_mean = 1,
            workers = cores,
        )

        model_dmm = Doc2Vec(
            corpus,
            dm = 1, # Distributed Memory Mean
            vector_size = int(n_features/2),
            dm_concat = 0,
            dm_mean = 1,
            workers = cores,
        )
        
        model = ConcatenatedDoc2Vec([model_dbow, model_dmm])

        X = [model.dv[str(doc.tags[0])] for doc in corpus]
        print(len(X[0]))  
        y = df['category'].astype('str')
            

        for algorithm in algorithms.keys():
            # Validation croisée 
            predictions = cross_val_predict(algorithms[algorithm], X, y, cv=stratified_kfold)
            result = classification_report(y, predictions, output_dict=True)

            # Performances (rappel, précision, score F)
            results = {
                '% Incels' : int(ratio),
                'Algorithme' : algorithm,
                'Modèle de vectorisation': 'Distributed Representations (Doc2Vec)',
                'N traits discr.' : n_features,
                'accuracy': result['accuracy']    
            }
            results.update(result['macro avg'])
            report.append(results)
            
            print(algorithm, f' | {ratio}% Incels | ', f'{n_features} traits discr.\n', classification_report(y, predictions))
            clear_output(wait=True)

    # Exporter les résults
    # support = Nombre d'occurrence dans chaque classe
    report = pd.DataFrame(report)
    report['nb_posts_incels'] = (report['% Incels'].apply(lambda x: x/100) * report['support']).astype(int)
    report['nb_posts_neutral'] = (report['% Incels'].apply(lambda x: 1-(x/100)) * report['support']).astype(int)

    report = report.rename(columns={'support':'nb_posts_total'})
    report['nb_posts_total'] = report['nb_posts_total'].astype(int)

    # Réorganiser l'ordre des colonnes
    report = report[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme', 
                    'N traits discr.', 'accuracy', 'precision', 'recall','f1-score']]

    report.sort_values(by='f1-score', ascending=False).to_csv(
    f'../3-results/results_training_doc2vec_{ratio}pc.csv', index=False
    )

125


KeyboardInterrupt: 